In [17]:
import os
import logging
from langchain_community.document_loaders import UnstructuredCSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever
import ollama

# Configure logging
logging.basicConfig(level=logging.INFO)


In [18]:
DOC_PATH = "data/shortjokes.csv"
print(DOC_PATH)

data/small list.csv


In [19]:
MODEL_NAME = "llama3.2"
print(MODEL_NAME)

llama3.2


In [20]:
EMBEDDING_MODEL = "nomic-embed-text"
print(EMBEDDING_MODEL)
VECTOR_STORE_NAME = "simple-rag"
print(VECTOR_STORE_NAME)


nomic-embed-text
simple-rag


In [23]:
def create_retriever(vector_db, llm):
    """Create a multi-query retriever."""
    QUERY_PROMPT = PromptTemplate(
        input_variables=["question"],
        template=""" Your task is to entertain by making good funny stories,
        you have 40 years of experience in cracking good and funny jokes,
        you can entertain with lots of funny jokes.

Original question: {question}""",
    )

    retriever = MultiQueryRetriever.from_llm(
        vector_db.as_retriever(), llm, prompt=QUERY_PROMPT
    )
    logging.info("Retriever created.")
    return retriever


def create_chain(retriever, llm):
    """Create the chain"""
    # RAG prompt
    template = """Answer the question based ONLY on the following context:
{context}
Question: {question}
"""

    prompt = ChatPromptTemplate.from_template(template)

    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    logging.info("Chain created successfully.")
    return chain


def main():
    # Load and process the CSV document
    data = ingest_csv(DOC_PATH)
    if data is None:
        return

    # Split the documents into chunks
    chunks = split_documents(data)

    # Create the vector database
    vector_db = create_vector_db(chunks)

    # Initialize the language model
    llm = ChatOllama(model=MODEL_NAME)

    # Create the retriever
    retriever = create_retriever(vector_db, llm)

    # Create the chain with preserved syntax
    chain = create_chain(retriever, llm)

    # Example query
    question = "what's up ?"

    # Get the response
    res = chain.invoke(input=question)
    print("Response:")
    print(res)


if __name__ == "__main__":
    main()

INFO:root:PDF loaded successfully.
INFO:root:Documents split into chunks.


[Document(metadata={'source': 'data/small list.csv'}, page_content='Name VUID Safetap Apollo PVC Safety Shoes (6 no) CP-023470-1 Safetap Apollo PVC Safety Shoes (7 no) CP-023470-2 Metal/Steel Cutting Blade (Size: 14 inch) CP-218683-2 Spades/Pawda Regular 8 No. without Handle (Size: 8 no.) CP-522929-1 Bitumen (Damar) 9015 CP-058941-1 Wooden Taar/Wire Brush CP-038739-1 Kantan Hessian Cloth / Jute Brown (Width 1 Meter) CP-049621-1 Measuring Tape (Steel) (5 meter) CP-796865-1 Measuring Tape (Steel) (30 meter) CP-796865-2 16 AMP 5 Pin 3 Phase Industrial Plug CP-968013-1 Shovel with Handle CP-740924-1')]
[Document(metadata={'source': 'data/small list.csv'}, page_content='Name VUID Safetap Apollo PVC Safety Shoes (6 no) CP-023470-1 Safetap Apollo PVC Safety Shoes (7 no) CP-023470-2 Metal/Steel Cutting Blade (Size: 14 inch) CP-218683-2 Spades/Pawda Regular 8 No. without Handle (Size: 8 no.) CP-522929-1 Bitumen (Damar) 9015 CP-058941-1 Wooden Taar/Wire Brush CP-038739-1 Kantan Hessian Cloth / J

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/pull "HTTP/1.1 200 OK"


TypeError: 'NoneType' object is not iterable